# Real video: Qwen2.5-VL on NExT-QA frames (n=240, per-type breakdown)

End-to-end **video** measurement with the vmd harness: real frames from NExT-QA clips, Qwen2.5-VL-3B, and frame-sequence corruptions. Produces:

- **Collapse gap** — frames vs. no frames (language prior).
- **Temporal-order effect** — frames shuffled (content preserved, order destroyed).
- **Content effect** — frames dropped.
- **Per-type breakdown** — causal (C) / temporal (T) / descriptive (D) questions: ¿dónde duele el shuffle?

**Runtime**: ~6 configs × 240 items ≈ 1440 llamadas de inferencia. ~50–70 min en L4, menos en A100. Cada configuración se guarda al completarse (checkpoint): si se corta, re-ejecuta la celda del grid y retoma.

> ⚠️ Celdas en orden, idempotentes. Si algo queda raro: *Entorno de ejecución → Reiniciar*.

In [1]:
# Setup idempotente
import os, shutil, sys
REPO = "/content/video-modality-diagnostics"
os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q https://github.com/mlahozy21/video-modality-diagnostics.git
os.chdir(REPO)
!pip install -q -e ".[video]"
for k in list(sys.modules):
    if k.startswith("vmd"):
        del sys.modules[k]
sys.path.insert(0, f"{REPO}/src")
import torch
print("setup OK —", os.getcwd(), "| CUDA:", torch.cuda.is_available())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for video-modality-diagnostics (pyproject.toml) ... done
setup OK — /content/video-modality-diagnostics | CUDA: True


In [2]:
# Subconjunto real de NExT-QA (30 preguntas por cada uno de los 8 tipos)
N = 240
!python scripts/prepare_nextqa.py --n {N} --out-dir data/nextqa

MC/test-00000-of-00001.parquet: 100% 899k/899k [00:01<00:00, 617kB/s]
MC split: 8564 questions, columns: ['video', 'frame_count', 'width', 'height', 'question', 'answer', 'qid', 'type', 'a0', 'a1', 'a2', 'a3', 'a4']
/content/video-modality-diagnostics/scripts/prepare_nextqa.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(per_type, len(g)),
sampled 240 questions over types: {'CH': 30, 'CW': 30, 'DC': 30, 'DL': 30, 'DO': 30, 'TC': 30, 'TN': 30, 'TP': 30}
videos.zip: 100% 6.49G/6.49G [00:25<00:00, 252MB/s]
wrote 240 items to data/nextqa/videoqa.jsonl


In [3]:
import json, os

from vmd.data import load_jsonl
from vmd.modalities import make_view
from vmd.video import VLMVideoBackend

items = load_jsonl("data/nextqa/videoqa.jsonl")
print(f"{len(items)} items de NExT-QA con vídeo real")

def qtype(item):
    return item.id.split("_")[1]          # CW/CH/TN/TC/TP/DC/DL/DO

def qgroup(item):
    return qtype(item)[0]                 # C / T / D

MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

# configuracion -> (canales, severidad, tipo de corrupcion)
CONFIGS = {
    "full":        (["vision"], 0.0,  "shuffle"),
    "blind":       ([],         0.0,  "shuffle"),
    "shuffle_0.5": (["vision"], 0.5,  "shuffle"),
    "shuffle_1.0": (["vision"], 1.0,  "shuffle"),
    "drop_0.5":    (["vision"], 0.5,  "drop"),
    "drop_0.75":   (["vision"], 0.75, "drop"),
}

240 items de NExT-QA con vídeo real


## Grid de evaluación (con checkpoint por configuración)

In [4]:
CKPT = "records_video.json"
records = json.load(open(CKPT)) if os.path.exists(CKPT) else {}

backend = VLMVideoBackend(MODEL, n_frames=8)

for cfg, (mods, sev, kind) in CONFIGS.items():
    if cfg in records:
        print(f"{cfg}: ya hecho ({len(records[cfg])} items), salto")
        continue
    backend.corruption_kind = kind
    preds = {}
    for j, it in enumerate(items):
        view = make_view(it, mods, corrupt={"vision": sev} if sev else None,
                         kind="shuffle", seed=0)   # corrupcion textual no aplica (evidence vacia)
        preds[it.id] = int(backend.answer(it, view) == it.answer_idx)
        if (j + 1) % 60 == 0:
            print(f"  {cfg}: {j+1}/{len(items)}")
    records[cfg] = preds
    json.dump(records, open(CKPT, "w"))
    acc = sum(preds.values()) / len(preds)
    print(f"{cfg}: accuracy {acc:.3f}  [checkpoint guardado]")

print("\ngrid completo:", list(records))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

  full: 60/240
  full: 120/240
  full: 180/240
  full: 240/240
full: accuracy 0.612  [checkpoint guardado]
  blind: 60/240
  blind: 120/240
  blind: 180/240
  blind: 240/240
blind: accuracy 0.412  [checkpoint guardado]
  shuffle_0.5: 60/240
  shuffle_0.5: 120/240
  shuffle_0.5: 180/240
  shuffle_0.5: 240/240
shuffle_0.5: accuracy 0.612  [checkpoint guardado]
  shuffle_1.0: 60/240
  shuffle_1.0: 120/240
  shuffle_1.0: 180/240
  shuffle_1.0: 240/240
shuffle_1.0: accuracy 0.635  [checkpoint guardado]
  drop_0.5: 60/240
  drop_0.5: 120/240
  drop_0.5: 180/240
  drop_0.5: 240/240
drop_0.5: accuracy 0.576  [checkpoint guardado]
  drop_0.75: 60/240
  drop_0.75: 120/240
  drop_0.75: 180/240
  drop_0.75: 240/240
drop_0.75: accuracy 0.612  [checkpoint guardado]

grid completo: ['full', 'blind', 'shuffle_0.5', 'shuffle_1.0', 'drop_0.5', 'drop_0.75']


## Tablas para el README

In [5]:
import numpy as np

def acc(cfg, group=None):
    sel = [it for it in items if group is None or qgroup(it) == group]
    vals = [records[cfg][it.id] for it in sel if it.id in records[cfg]]
    return float(np.mean(vals)), len(vals)

print(f"| Metric (Qwen2.5-VL-3B, NExT-QA n={len(items)}) | Accuracy |")
print("|---|---:|")
labels = {"full": "Full accuracy (8 frames)", "blind": "Blind (no frames)",
          "shuffle_0.5": "Shuffle 50% of frames", "shuffle_1.0": "Shuffle 100% of frames",
          "drop_0.5": "Drop 50% of frames", "drop_0.75": "Drop 75% of frames"}
for cfg, lab in labels.items():
    print(f"| {lab} | {acc(cfg)[0]:.3f} |")
print(f"| **Collapse gap** | **{acc('full')[0] - acc('blind')[0]:.3f}** |")

print("\nPer question group (C=causal, T=temporal, D=descriptive):\n")
print("| Group | n | Full | Blind | Gap | Shuffle 100% | Δ shuffle | Drop 75% | Δ drop |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for g, name in [("C", "Causal"), ("T", "Temporal"), ("D", "Descriptive")]:
    full, n = acc("full", g)
    blind = acc("blind", g)[0]
    sh = acc("shuffle_1.0", g)[0]
    dr = acc("drop_0.75", g)[0]
    print(f"| {name} | {n} | {full:.3f} | {blind:.3f} | {full-blind:+.3f} "
          f"| {sh:.3f} | {sh-full:+.3f} | {dr:.3f} | {dr-full:+.3f} |")

| Metric (Qwen2.5-VL-3B, NExT-QA n=240) | Accuracy |
|---|---:|
| Full accuracy (8 frames) | 0.625 |
| Blind (no frames) | 0.433 |
| Shuffle 50% of frames | 0.625 |
| Shuffle 100% of frames | 0.650 |
| Drop 50% of frames | 0.592 |
| Drop 75% of frames | 0.600 |
| **Collapse gap** | **0.192** |

Per question group (C=causal, T=temporal, D=descriptive):

| Group | n | Full | Blind | Gap | Shuffle 100% | Δ shuffle | Drop 75% | Δ drop |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Causal | 60 | 0.600 | 0.400 | +0.200 | 0.650 | +0.050 | 0.567 | -0.033 |
| Temporal | 90 | 0.511 | 0.478 | +0.033 | 0.533 | +0.022 | 0.511 | +0.000 |
| Descriptive | 90 | 0.756 | 0.411 | +0.344 | 0.767 | +0.011 | 0.711 | -0.044 |


## Cómo leerlo

- **Gap por grupo**: ¿en qué tipo de pregunta mira más el vídeo?
- **Δ shuffle en temporales**: si el modelo usara el orden, el shuffle debería doler *sobre todo* en T. Si Δ ≈ 0 también ahí, el sesgo bag-of-frames es fuerte incluso donde el orden es la pregunta.
- **Δ drop**: cuánto contenido necesita por grupo.

Descarga `records_video.json` antes de cerrar (panel Archivos) — contiene la predicción por item y permite cualquier análisis posterior sin re-inferencia.